# Qwen2.5-7B-Instruct LoRA Fine-tuning — Robot Task Planner

**Pipeline:** Load dataset → 4-bit model → LoRA → SFTTrainer → merge → GGUF → Ollama

**Bugs fixed vs previous version:**
- Removed `torch.compile`: not supported on bitsandbytes quantized models
- `fp16=True`: was False; must match `bnb_4bit_compute_dtype=float16`. T4 does not support bf16.
- `optim=paged_adamw_8bit`: was `adamw_torch` which OOMs on T4 with 7B
- `per_device_train_batch_size=2` + `grad_accum=8`: batch=4 risks OOM
- Removed `dataset.map(preprocess)`: was silently truncating outputs at 256 tokens
- Removed `tokenizer.bos_token = tokenizer.eos_token`: corrupts token config
- Fixed `torchdtype=` typo in inference cell (was causing float32 OOM)

In [ ]:
# Cell 1: Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "Enable GPU in Kaggle settings."

In [ ]:
# Cell 2: Install
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [ ]:
# Cell 3: Config
import json
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_NAME   = "Qwen/Qwen2.5-7B-Instruct"
DATASET_PATH = "/kaggle/input/morpho-robot-planner-dataset/dataset.json"  # update if needed
OUTPUT_DIR   = "/kaggle/working/qwen_robot_lora"
MERGED_DIR   = "/kaggle/working/qwen_robot_merged"

print(f"Model  : {MODEL_NAME}")
print(f"Dataset: {DATASET_PATH}")

## 1. Load Dataset

In [ ]:
# Cell 4
with open(DATASET_PATH, "r") as f:
    raw = json.load(f)

dataset = Dataset.from_list(raw)
print(f"Samples : {len(dataset)}")
print(f"Columns : {dataset.column_names}")
assert "text" in dataset.column_names, "Dataset must have a 'text' column in ChatML format."

# Sanity check token length
from transformers import AutoTokenizer as _T
_tok = _T.from_pretrained(MODEL_NAME, trust_remote_code=True)
sample_len = len(_tok(dataset[0]["text"]).input_ids)
print(f"Sample[0] token length: {sample_len}  (must be < 512)")
del _tok

## 2. Tokenizer

In [ ]:
# Cell 5
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

# FIX: do NOT overwrite bos_token with eos_token
# The library aligns token IDs automatically; manual override corrupts generation

print(f"Pad: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"BOS: {tokenizer.bos_token} (id={tokenizer.bos_token_id})")
print(f"EOS: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")

## 3. Load Model (4-bit)

In [ ]:
# Cell 6
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,  # must match fp16=True in SFTConfig
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1

dtypes = {}
for _, p in model.named_parameters():
    dtypes[p.dtype] = dtypes.get(p.dtype, 0) + 1
print(f"Param dtypes: {dtypes}")
# Expect: uint8 (quantized weights), float16 (norms + embeddings)

## 4. Apply LoRA

In [ ]:
# Cell 7
# FIX: prepare_model_for_kbit_training required before LoRA on quantized model
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~40M trainable / 7.6B total (~0.5%)

## 5. Train

In [ ]:
# Cell 8
# FIX summary:
#   fp16=True           : was False; must match bnb compute dtype (float16)
#   paged_adamw_8bit    : was adamw_torch which keeps fp32 optimizer states in VRAM -> OOM
#   batch_size=2        : was 4; 7B on T4 (15.6GB) is tight; grad_accum=8 keeps effective=16
#   NO dataset.map()    : was silently truncating all samples at 256 tokens
#   NO torch.compile    : bitsandbytes quantized models are not supported by torch.compile

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,       # effective batch = 16
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    fp16=True,                           # T4 is Turing; no bf16 support
    bf16=False,
    max_seq_length=512,
    dataset_text_field="text",           # SFTTrainer tokenizes here; do NOT pre-tokenize
    dataloader_num_workers=2,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
)

steps_per_epoch = len(dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
print(f"Samples        : {len(dataset)}")
print(f"Steps / epoch  : {steps_per_epoch}")
print(f"Total steps    : {steps_per_epoch * training_args.num_train_epochs}")
print("Training...")

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved -> {OUTPUT_DIR}")

## 6. Merge LoRA into Base

In [ ]:
# Cell 9
import gc
from peft import PeftModel

torch.cuda.empty_cache()
gc.collect()

print("Loading base model on CPU for merge...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="cpu", trust_remote_code=True
)

print("Loading adapter...")
peft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

print("Merging...")
merged = peft_model.merge_and_unload()

print(f"Saving -> {MERGED_DIR}")
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

print("Done.")
del base_model, peft_model, merged
torch.cuda.empty_cache()
gc.collect()

## 7. Inference Test

In [ ]:
# Cell 10
# FIX: was `torchdtype=torch.float16` (typo) -> model loaded in float32 -> OOM
test_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.float16,   # correct spelling
    device_map="auto"
)
test_tok = AutoTokenizer.from_pretrained(MERGED_DIR)

messages = [
    {"role": "system", "content": "You are a robot task planner. Output a JSON action plan only."},
    {"role": "user",   "content": 'Task: Go to the red_box.\nScene: {"objects": ["red_box", "blue_sphere"], "obstacles": ["wall"]}'}
]

text   = test_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = test_tok(text, return_tensors="pt").to(test_model.device)

with torch.no_grad():
    out = test_model.generate(
        **inputs, max_new_tokens=200, temperature=0.1,
        do_sample=True, pad_token_id=test_tok.pad_token_id
    )

response = test_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Output:")
print(response)

try:
    print("\nParsed JSON:")
    print(json.dumps(json.loads(response.strip()), indent=2))
except json.JSONDecodeError as e:
    print(f"Invalid JSON: {e} — may need more epochs or dataset review.")

## 8. GGUF Conversion + Ollama (run LOCALLY after downloading merged model)

```bash
# 1. Clone llama.cpp
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
pip install -r requirements.txt

# 2. Convert to GGUF Q4_K_M
python convert_hf_to_gguf.py /path/to/qwen_robot_merged \
    --outfile qwen_robot_planner.gguf \
    --outtype q4_k_m

# 3. Modelfile
cat > Modelfile << 'EOF'
FROM ./qwen_robot_planner.gguf
SYSTEM "You are a robot task planner. Given a task and scene, produce a JSON action plan only. No explanation."
PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"
EOF

# 4. Register
ollama create qwen-robot-planner -f Modelfile

# 5. Test
ollama run qwen-robot-planner 'Task: Go to red_box. Scene: {"objects":["red_box"],"obstacles":["wall"]}'
```

### ROS2 node integration
```python
import ollama, json

def call_llm_planner(task: str, scene: dict) -> dict:
    resp = ollama.chat(
        model="qwen-robot-planner",
        messages=[{"role": "user", "content": f'Task: {task}\nScene: {json.dumps(scene)}'}]
    )
    return json.loads(resp["message"]["content"].strip())
```